In [ ]:
#@title Cell 1 - Notebook overview

from IPython.display import display, Markdown

display(Markdown(r"""
# Simulation 01: Complete chromosome–plasmid design — updated biological specification

## Central question

\[
\boxed{\text{Does the effect of plasmid }j\text{ depend on the pathogen chromosome }i?}
\]

The simulated model is

\[
y_{ij}
=
\alpha
+
\boldsymbol{\beta}_C^T\mathbf c_i
+
\boldsymbol{\beta}_P^T\mathbf p_j
+
\mathbf c_i^T\mathbf B\mathbf p_j
+
u_i
+
\varepsilon_{ij}.
\]

The change in MIC caused by plasmid \(j\) in pathogen \(i\) is

\[
\Delta_{ij}=y_{ij}-y_{i0},
\]

and the chromosome-dependent difference in that MIC change is

\[
\Delta\Delta_{ik,j}=\Delta_{ij}-\Delta_{kj}.
\]

## Agreed Scenario 1 design

- 200 pathogen chromosomal backgrounds.
- 20 separate blaTEM-1-carrying plasmids.
- Each pathogen is also observed in the plasmid-free state \(P_0\).
- Complete \(200\times21=4200\) MIC observations.
- 30 gene-centred chromosomal units; each has one coding state and one paired 300-bp putative promoter-region state.
- Each chromosomal state is in \(\{-1,0,+1\}\) with probabilities \(0.15,0.70,0.15\).
- 500 additional background SNPs are used only to construct \(K\).
- TEM-1 amino-acid sequence is held constant.
- Each simulated plasmid receives an empirical promoter-genotype/copy-number pair sampled from the same empirical pathogen record.
- Promoter effects: C32T \(=+0.50\), G162T \(=+0.25\), G175A \(=0\) log2 MIC.
- Other promoter-region mutations are retained in the sampled empirical record but have zero direct effect.
- Copy number is represented as
  \[
  q_j=\log_2\!\left(\frac{CN_j}{\operatorname{median}(CN_{\mathrm{TEM1}})}\right).
  \]
- Reference TEM-1 plasmid effect at median copy number: \(+0.50\) log2 MIC.
- Copy-number coefficient: \(+0.25\) per doubling relative to the empirical median.
- Plasmid feature vector:
  \[
  \mathbf p_j=(1,\ I(C32T),\ I(G162T),\ I(G175A),\ q_j)^T.
  \]
- \(P_0\) is the all-zero plasmid vector.
- Chromosome–plasmid interaction is restricted in the data-generating model to efflux machinery/regulators and OmpF/OmpC permeability.
- Only the plasmid-presence component participates in the chromosome–plasmid interaction.
- Interaction magnitude: \(0.10\) log2 MIC per relevant chromosomal state.
- \(\alpha=-2.5\).
- \(\sigma_g=0.40\), \(\sigma_e=0.32\).

The 100 simulation replicates and 200 representative parametric-bootstrap replicates are retained from the previous notebook as evaluation settings; they are not biological parameters.
"""))

print("Transition: Cell 2 loads the public plasmid-feature input and defines the fixed simulation settings.")


In [ ]:
#@title Cell 2 - Load public plasmid-feature input and define fixed settings

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize_scalar

REPO_ROOT = Path.cwd()
DATA_FILE = REPO_ROOT / "data" / "plasmid_feature_pairs.csv"

OUTPUT_DIR = (
    REPO_ROOT
    / "results"
    / "simulation_01_complete_design"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Public plasmid-feature input was not found:\n"
        f"{DATA_FILE}\n"
        "Run the notebook from the repository root."
    )

empirical_pairs = pd.read_csv(
    DATA_FILE,
    low_memory=False,
)

KEY_SITE_COLUMNS = [
    "sutcliffe_32_nt",
    "sutcliffe_162_nt",
    "sutcliffe_175_nt",
]

required_columns = [
    *KEY_SITE_COLUMNS,
    "CN_TEM1",
]

missing_columns = [
    c for c in required_columns
    if c not in empirical_pairs.columns
]

if missing_columns:
    raise RuntimeError(
        "The public plasmid-feature input is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

empirical_pairs = empirical_pairs[
    required_columns
].copy()

for col in KEY_SITE_COLUMNS:
    empirical_pairs[col] = (
        empirical_pairs[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

empirical_pairs["CN_TEM1"] = pd.to_numeric(
    empirical_pairs["CN_TEM1"],
    errors="coerce",
)

valid_nt = {"A", "C", "G", "T"}

complete_mask = (
    empirical_pairs[KEY_SITE_COLUMNS]
    .apply(lambda s: s.isin(valid_nt))
    .all(axis=1)
    & empirical_pairs["CN_TEM1"].notna()
    & (empirical_pairs["CN_TEM1"] > 0)
)

empirical_pairs = (
    empirical_pairs.loc[complete_mask]
    .reset_index(drop=True)
)

if len(empirical_pairs) < 20:
    raise RuntimeError(
        f"Only {len(empirical_pairs)} complete promoter/copy-number pairs remain; "
        "at least 20 are required."
    )

CN_REFERENCE = float(
    empirical_pairs["CN_TEM1"].median()
)

if not np.isfinite(CN_REFERENCE) or CN_REFERENCE <= 0:
    raise RuntimeError(
        "The median CN_TEM1 is not positive."
    )

# -------------------------------------------------------------------------
# Fixed Scenario 1 settings
# -------------------------------------------------------------------------

MASTER_SEED = 20260906

N_PATHOGENS = 200
N_PLASMIDS = 20

PREDEFINED_GENES = [
    "acrB", "acrR", "ampC", "basR", "cirA", "cyaA", "fabI", "folP", "ftsI",
    "gyrA", "marR", "nfsA", "nfsB", "ompC", "ompF", "parC", "parE", "pmrB",
    "ptsI", "rpoB", "rpsL", "soxR", "soxS", "uhpT", "acrA", "tolC", "marA",
    "rob", "ompR", "envZ",
]

if len(PREDEFINED_GENES) != 30:
    raise ValueError("The predefined chromosomal gene panel must contain exactly 30 genes.")

TARGET_FEATURE_LABELS = []
for gene in PREDEFINED_GENES:
    TARGET_FEATURE_LABELS.extend([
        f"{gene}:coding",
        f"{gene}:upstream_300bp",
    ])

D_C = len(TARGET_FEATURE_LABELS)  # 60

PLASMID_FEATURE_LABELS = [
    "TEM1_plasmid_presence",
    "C32T",
    "G162T",
    "G175A",
    "log2_CN_relative_to_empirical_median",
]
D_P = len(PLASMID_FEATURE_LABELS)  # 5

CHROMOSOMAL_STATES = np.array([-1.0, 0.0, 1.0])
CHROMOSOMAL_STATE_PROBS = np.array([0.15, 0.70, 0.15])

N_BACKGROUND_SNPS = 500
ALLELE_FREQ_LOW = 0.10
ALLELE_FREQ_HIGH = 0.90

ALPHA_TRUE = -2.5

SIGMA_G_TRUE = 0.40
SIGMA_E_TRUE = 0.32
SIGMA_G2_TRUE = SIGMA_G_TRUE ** 2
SIGMA_E2_TRUE = SIGMA_E_TRUE ** 2

BETA_P_TRUE = np.array([
    0.50,  # reference TEM-1 plasmid at median CN
    0.50,  # C32T
    0.25,  # G162T
    0.00,  # G175A
    0.25,  # one doubling of CN relative to empirical median
], dtype=float)

INTERACTION_MAGNITUDE = 0.10

N_SIM_REPLICATES = 100
N_BOOTSTRAP = 200

# Optional 100 x 200 repeated-dataset bootstrap coverage.
RUN_FULL_BOOTSTRAP_COVERAGE = False

print("=" * 90)
print("SIMULATION 01 — UPDATED FIXED SETTINGS")
print("=" * 90)
print(f"Empirical promoter/CN pairs available: {len(empirical_pairs):,}")
print(f"Empirical median CN_TEM1:               {CN_REFERENCE:.6f}")
print(f"Pathogens:                              {N_PATHOGENS}")
print(f"Plasmids:                               {N_PLASMIDS}")
print(f"Plasmid states including P0:            {N_PLASMIDS + 1}")
print(f"Complete MIC observations:              {N_PATHOGENS * (N_PLASMIDS + 1):,}")
print(f"Gene-centred units:                     {len(PREDEFINED_GENES)}")
print(f"Chromosomal features:                   {D_C}")
print(f"Background SNPs for K:                  {N_BACKGROUND_SNPS}")
print(f"Plasmid features:                       {D_P}")
print(f"alpha:                                  {ALPHA_TRUE}")
print(f"sigma_g:                                {SIGMA_G_TRUE}")
print(f"sigma_e:                                {SIGMA_E_TRUE}")
print(f"Output directory:                       {OUTPUT_DIR}")
print("\nCell 2: PASS")


In [ ]:
#@title Cell 3 - Define the updated biological simulation functions

def feature_index(gene, region):
    label = (
        f"{gene}:coding"
        if region == "coding"
        else f"{gene}:upstream_300bp"
    )
    return TARGET_FEATURE_LABELS.index(label)


def build_true_chromosomal_coefficients():
    """
    Construct the agreed fixed chromosome main effects and sparse
    chromosome x TEM-1-plasmid interaction matrix.
    """
    beta_C = np.zeros(D_C, dtype=float)
    B = np.zeros((D_C, D_P), dtype=float)

    efflux_machinery = {"acrA", "acrB", "tolC"}
    repressors = {"acrR", "marR"}
    activators = {"marA", "rob", "soxR", "soxS"}
    porins = {"ompC", "ompF"}

    for gene in PREDEFINED_GENES:
        coding_i = feature_index(gene, "coding")
        upstream_i = feature_index(gene, "upstream")

        # Chromosomal main effects.
        if gene in efflux_machinery:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in repressors:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene in activators:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in porins:
            # +1 means greater porin function/expression -> lower CAZ MIC.
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene == "ampC":
            beta_C[coding_i] = +0.10
            beta_C[upstream_i] = +0.25

        elif gene == "ftsI":
            # +1 coding state means increased ceftazidime binding -> lower MIC.
            beta_C[coding_i] = -0.25
            # +1 upstream state means increased expression; keep the effect weak.
            beta_C[upstream_i] = -0.10

        # ompR, envZ, cirA and the remaining genes have zero direct
        # CAZ effect in Scenario 1.

        # Sparse chromosome x plasmid interaction.
        # Only the TEM-1 plasmid-presence position participates.
        if gene in efflux_machinery:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in repressors:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

        elif gene in activators:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in porins:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

    return beta_C, B


BETA_C_TRUE, B_TRUE = build_true_chromosomal_coefficients()


def simulate_targeted_chromosomal_features(rng):
    """
    Generate 60 linked-by-label ternary chromosomal states:
    one coding and one paired 300-bp-upstream state for each of 30 genes.

    Each state:
        -1 with probability 0.15
         0 with probability 0.70
        +1 with probability 0.15
    """
    for _ in range(100):
        C = rng.choice(
            CHROMOSOMAL_STATES,
            size=(N_PATHOGENS, D_C),
            p=CHROMOSOMAL_STATE_PROBS,
        ).astype(float)

        augmented = np.column_stack([
            np.ones(N_PATHOGENS, dtype=float),
            C,
        ])

        if np.linalg.matrix_rank(augmented) == D_C + 1:
            return C

    raise RuntimeError(
        "Could not generate a full-rank pathogen chromosome matrix after 100 attempts."
    )


def sample_empirical_plasmids(rng):
    """
    Sample 20 empirical promoter-genotype/CN pairs without replacement.

    The promoter genotype and CN_TEM1 always come from the same empirical
    pathogen record. Sampling is repeated only if needed to give a full-rank
    five-feature plasmid representation.
    """
    n_empirical = len(empirical_pairs)

    for _ in range(5000):
        selected_positions = rng.choice(
            n_empirical,
            size=N_PLASMIDS,
            replace=False,
        )

        sampled = (
            empirical_pairs.iloc[selected_positions]
            .copy()
            .reset_index(drop=True)
        )

        sampled.insert(
            0,
            "plasmid_id",
            [f"P{j}" for j in range(1, N_PLASMIDS + 1)],
        )

        sampled["I_C32T"] = (
            sampled["sutcliffe_32_nt"].eq("T")
        ).astype(float)

        sampled["I_G162T"] = (
            sampled["sutcliffe_162_nt"].eq("T")
        ).astype(float)

        sampled["I_G175A"] = (
            sampled["sutcliffe_175_nt"].eq("A")
        ).astype(float)

        sampled["q_CN"] = np.log2(
            sampled["CN_TEM1"].astype(float)
            / CN_REFERENCE
        )

        P = np.column_stack([
            np.ones(N_PLASMIDS, dtype=float),
            sampled["I_C32T"].to_numpy(dtype=float),
            sampled["I_G162T"].to_numpy(dtype=float),
            sampled["I_G175A"].to_numpy(dtype=float),
            sampled["q_CN"].to_numpy(dtype=float),
        ])

        P_all = np.vstack([
            np.zeros((1, D_P), dtype=float),
            P,
        ])

        augmented_state_matrix = np.column_stack([
            np.ones(N_PLASMIDS + 1, dtype=float),
            P_all,
        ])

        if np.linalg.matrix_rank(augmented_state_matrix) == D_P + 1:
            return P, sampled

    raise RuntimeError(
        "Could not sample 20 empirical plasmid profiles giving a full-rank "
        "plasmid feature matrix after 5000 attempts."
    )


def simulate_background_relatedness(rng):
    """
    Retain the previous Scenario 1 construction of K:
    500 biallelic background SNPs outside the 30 targeted gene-centred units.

        K = Z Z^T / sum_m p_m(1-p_m)
        Z_im = x_im - p_m
    """
    source_frequencies = rng.uniform(
        ALLELE_FREQ_LOW,
        ALLELE_FREQ_HIGH,
        size=N_BACKGROUND_SNPS,
    )

    G = rng.binomial(
        1,
        source_frequencies,
        size=(N_PATHOGENS, N_BACKGROUND_SNPS),
    ).astype(float)

    p = G.mean(axis=0)
    Z = G - p[None, :]

    denominator = float(
        np.sum(p * (1.0 - p))
    )

    if denominator <= 0:
        raise ValueError(
            "Background-SNP relatedness denominator is not positive."
        )

    K = (Z @ Z.T) / denominator
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)

    if eigenvalues.min() < -1e-8:
        raise ValueError(
            "Constructed K is unexpectedly non-PSD: "
            f"minimum eigenvalue={eigenvalues.min():.6g}"
        )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    K = (
        eigenvectors * eigenvalues
    ) @ eigenvectors.T

    K = (K + K.T) / 2.0

    return K, G, p


def build_complete_design(C, P):
    """
    Build every pathogen in every plasmid state.

    P0 is the all-zero plasmid vector.
    States P1...P20 each contain one sampled TEM-1 plasmid profile.
    """
    P_all = np.vstack([
        np.zeros((1, D_P), dtype=float),
        P,
    ])

    n_states = N_PLASMIDS + 1

    pathogen_index = np.repeat(
        np.arange(N_PATHOGENS, dtype=int),
        n_states,
    )

    plasmid_state_index = np.tile(
        np.arange(n_states, dtype=int),
        N_PATHOGENS,
    )

    C_obs = C[pathogen_index, :]
    P_obs = P_all[plasmid_state_index, :]

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(pathogen_index),
        D_C * D_P,
    )

    X = np.column_stack([
        np.ones(len(pathogen_index), dtype=float),
        C_obs,
        P_obs,
        interaction,
    ])

    return (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    )


def draw_correlated_host_effect(rng, K, sigma_g2):
    eigenvalues, eigenvectors = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    z = rng.normal(
        0.0,
        1.0,
        size=N_PATHOGENS,
    )

    u = eigenvectors @ (
        np.sqrt(
            sigma_g2 * eigenvalues
        ) * z
    )

    return u


def simulate_complete_dataset(seed):
    rng = np.random.default_rng(seed)

    C = simulate_targeted_chromosomal_features(rng)
    P, sampled_plasmids = sample_empirical_plasmids(rng)

    (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    ) = build_complete_design(C, P)

    expected_columns = (
        1
        + D_C
        + D_P
        + D_C * D_P
    )

    if X.shape[1] != expected_columns:
        raise RuntimeError(
            f"Unexpected fixed-effect column count: {X.shape[1]} "
            f"instead of {expected_columns}."
        )

    K, G_background, background_frequencies = (
        simulate_background_relatedness(rng)
    )

    theta_true = np.concatenate([
        np.array([ALPHA_TRUE], dtype=float),
        BETA_C_TRUE,
        BETA_P_TRUE,
        B_TRUE.reshape(-1),
    ])

    structural_mean = X @ theta_true

    u = draw_correlated_host_effect(
        rng,
        K,
        SIGMA_G2_TRUE,
    )

    epsilon = rng.normal(
        0.0,
        SIGMA_E_TRUE,
        size=X.shape[0],
    )

    y = (
        structural_mean
        + u[pathogen_index]
        + epsilon
    )

    return {
        "seed": int(seed),
        "C": C,
        "P": P,
        "P_all": P_all,
        "sampled_plasmids": sampled_plasmids,
        "K": K,
        "G_background": G_background,
        "background_frequencies": background_frequencies,
        "beta_C_true": BETA_C_TRUE.copy(),
        "beta_P_true": BETA_P_TRUE.copy(),
        "B_true": B_TRUE.copy(),
        "theta_true": theta_true,
        "u_true": u,
        "epsilon_true": epsilon,
        "structural_mean": structural_mean,
        "X": X,
        "pathogen_index": pathogen_index,
        "plasmid_state_index": plasmid_state_index,
        "y": y,
    }


print("=" * 90)
print("UPDATED TRUE BIOLOGICAL COEFFICIENTS")
print("=" * 90)

chromosome_effect_table = pd.DataFrame({
    "feature": TARGET_FEATURE_LABELS,
    "beta_C": BETA_C_TRUE,
    "interaction_with_TEM1_presence": B_TRUE[:, 0],
})

plasmid_effect_table = pd.DataFrame({
    "plasmid_feature": PLASMID_FEATURE_LABELS,
    "beta_P": BETA_P_TRUE,
})

display(
    chromosome_effect_table[
        (chromosome_effect_table["beta_C"] != 0)
        | (chromosome_effect_table["interaction_with_TEM1_presence"] != 0)
    ].reset_index(drop=True)
)

display(plasmid_effect_table)

print(f"\nNon-zero beta_C coefficients: {np.count_nonzero(BETA_C_TRUE)}")
print(f"Non-zero B coefficients:      {np.count_nonzero(B_TRUE)}")
print("\nCell 3: PASS")


In [ ]:
#@title Cell 4 - Define efficient REML and GLS fitting

def prepare_reml_static(X, pathogen_index, K):
    """
    Prepare low-rank quantities for

        V = sigma_g^2 * K_obs + sigma_e^2 * I,

    where K_obs is pathogen-level K expanded to observation rows.

    This avoids constructing or inverting a 4200 x 4200 covariance matrix.
    """
    X = np.asarray(X, dtype=float)
    K = np.asarray(K, dtype=float)

    m, p_fixed = X.shape

    design_rank = np.linalg.matrix_rank(X)

    if design_rank != p_fixed:
        raise ValueError(
            f"Fixed-effect design matrix is rank deficient: "
            f"rank={design_rank}, columns={p_fixed}."
        )

    eigenvalues, Q = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    keep = eigenvalues > 1e-10
    eigenvalues = eigenvalues[keep]
    Q = Q[:, keep]

    # B_lowrank satisfies K_obs = B_lowrank B_lowrank^T.
    B_lowrank = (
        Q[pathogen_index, :]
        * np.sqrt(eigenvalues)[None, :]
    )

    static = {
        "m": int(m),
        "p_fixed": int(p_fixed),
        "rank_K": int(len(eigenvalues)),
        "B_lowrank": B_lowrank,
        "BtB": B_lowrank.T @ B_lowrank,
        "BtX": B_lowrank.T @ X,
        "XTX": X.T @ X,
        "X": X,
    }

    return static


def add_y_to_reml_static(static, y):
    working = {
        key: value
        for key, value in static.items()
        if key not in {"B_lowrank", "X"}
    }

    B_lowrank = static["B_lowrank"]
    X = static["X"]

    working["Bty"] = B_lowrank.T @ y
    working["Xty"] = X.T @ y
    working["yty"] = float(y @ y)

    return working


def evaluate_profile_reml(log_delta, working, return_fit=False):
    """
    Profile REML objective for

        delta = sigma_e^2 / sigma_g^2.
    """
    delta = float(np.exp(log_delta))

    m = working["m"]
    p_fixed = working["p_fixed"]
    rank_K = working["rank_K"]

    M = (
        np.eye(rank_K)
        + working["BtB"] / delta
    )

    try:
        chol_M = cho_factor(
            M,
            lower=True,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    M_inv_BtX = cho_solve(
        chol_M,
        working["BtX"],
        check_finite=False,
    )

    M_inv_Bty = cho_solve(
        chol_M,
        working["Bty"],
        check_finite=False,
    )

    XtAinvX = (
        working["XTX"] / delta
        - (
            working["BtX"].T
            @ M_inv_BtX
        ) / (delta ** 2)
    )

    XtAinvy = (
        working["Xty"] / delta
        - (
            working["BtX"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    yAinvy = (
        working["yty"] / delta
        - float(
            working["Bty"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    XtAinvX = (
        XtAinvX + XtAinvX.T
    ) / 2.0

    sign_X, logdet_X = np.linalg.slogdet(
        XtAinvX
    )

    if sign_X <= 0:
        return np.inf if not return_fit else None

    try:
        chol_X = cho_factor(
            XtAinvX,
            lower=True,
            check_finite=False,
        )

        beta_hat = cho_solve(
            chol_X,
            XtAinvy,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    q = float(
        yAinvy
        - beta_hat @ XtAinvy
    )

    df_reml = m - p_fixed

    if q <= 0 or df_reml <= 0:
        return np.inf if not return_fit else None

    logdet_M = 2.0 * np.sum(
        np.log(np.diag(chol_M[0]))
    )

    logdet_A = (
        m * np.log(delta)
        + logdet_M
    )

    objective = (
        logdet_A
        + logdet_X
        + df_reml * np.log(q / df_reml)
    )

    if return_fit:
        return {
            "objective": float(objective),
            "beta_hat": beta_hat,
            "q": q,
            "delta": delta,
            "df_reml": int(df_reml),
        }

    return float(objective)


def fit_section2_reml_gls(X, y, pathogen_index, K, static=None):
    if static is None:
        static = prepare_reml_static(
            X,
            pathogen_index,
            K,
        )

    working = add_y_to_reml_static(
        static,
        y,
    )

    optimization = minimize_scalar(
        lambda log_delta: evaluate_profile_reml(
            log_delta,
            working,
            return_fit=False,
        ),
        bounds=(-8.0, 8.0),
        method="bounded",
        options={
            "xatol": 1e-4,
            "maxiter": 100,
        },
    )

    if not optimization.success:
        raise RuntimeError(
            "REML optimization failed: "
            + str(optimization.message)
        )

    fit = evaluate_profile_reml(
        optimization.x,
        working,
        return_fit=True,
    )

    if fit is None:
        raise RuntimeError(
            "Final REML/GLS evaluation failed."
        )

    sigma_g2_hat = (
        fit["q"]
        / fit["df_reml"]
    )

    sigma_e2_hat = (
        fit["delta"]
        * sigma_g2_hat
    )

    return {
        "beta_hat": fit["beta_hat"],
        "sigma_g2_hat": float(sigma_g2_hat),
        "sigma_e2_hat": float(sigma_e2_hat),
        "delta_hat": float(fit["delta"]),
        "reml_objective": float(fit["objective"]),
        "optimization_nfev": int(optimization.nfev),
        "static": static,
    }


def unpack_beta(beta_hat):
    start_C = 1
    stop_C = start_C + D_C

    start_P = stop_C
    stop_P = start_P + D_P

    start_B = stop_P

    alpha_hat = float(beta_hat[0])
    beta_C_hat = beta_hat[start_C:stop_C]
    beta_P_hat = beta_hat[start_P:stop_P]
    B_hat = beta_hat[start_B:].reshape(
        D_C,
        D_P,
    )

    return (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    )


print("Cell 4: PASS")


In [ ]:
#@title Cell 5 - Generate, save, and QC one complete 200 x 21 MIC dataset

EXAMPLE_SEED = MASTER_SEED

example = simulate_complete_dataset(
    EXAMPLE_SEED
)

X = example["X"]
y = example["y"]
K = example["K"]
C = example["C"]
P = example["P"]
pathogen_index = example["pathogen_index"]

expected_rows = (
    N_PATHOGENS
    * (N_PLASMIDS + 1)
)

expected_columns = (
    1
    + D_C
    + D_P
    + D_C * D_P
)

if X.shape != (
    expected_rows,
    expected_columns,
):
    raise ValueError(
        f"Unexpected design matrix shape {X.shape}; "
        f"expected {(expected_rows, expected_columns)}."
    )

if K.shape != (
    N_PATHOGENS,
    N_PATHOGENS,
):
    raise ValueError(
        "K has the wrong dimensions."
    )

if np.max(
    np.abs(K - K.T)
) > 1e-10:
    raise ValueError(
        "K is not symmetric."
    )

K_eigenvalues = np.linalg.eigvalsh(K)

state_labels = [
    "P0",
    *[
        f"P{j}"
        for j in range(1, N_PLASMIDS + 1)
    ],
]

pathogen_labels = [
    f"C{i}"
    for i in range(1, N_PATHOGENS + 1)
]

observed_log2_matrix = pd.DataFrame(
    y.reshape(
        N_PATHOGENS,
        N_PLASMIDS + 1,
    ),
    index=pathogen_labels,
    columns=state_labels,
)

observed_mg_l_matrix = (
    2.0 ** observed_log2_matrix
)

structural_log2_matrix = pd.DataFrame(
    example["structural_mean"].reshape(
        N_PATHOGENS,
        N_PLASMIDS + 1,
    ),
    index=pathogen_labels,
    columns=state_labels,
)

chromosome_table = pd.DataFrame(
    C,
    columns=TARGET_FEATURE_LABELS,
)
chromosome_table.insert(
    0,
    "pathogen_id",
    pathogen_labels,
)

plasmid_table = example[
    "sampled_plasmids"
].copy()

for feature_name, values in zip(
    PLASMID_FEATURE_LABELS,
    P.T,
):
    plasmid_table[
        feature_name
    ] = values

OBSERVED_LOG2_PATH = (
    OUTPUT_DIR
    / "01_example_observed_log2_MIC_200x21.csv"
)

OBSERVED_MG_L_PATH = (
    OUTPUT_DIR
    / "01_example_observed_MIC_mg_L_200x21.csv"
)

STRUCTURAL_LOG2_PATH = (
    OUTPUT_DIR
    / "01_example_structural_log2_MIC_200x21.csv"
)

CHROMOSOME_PATH = (
    OUTPUT_DIR
    / "01_example_chromosomal_states.csv"
)

PLASMID_PATH = (
    OUTPUT_DIR
    / "01_example_sampled_empirical_plasmid_profiles.csv"
)

observed_log2_matrix.to_csv(
    OBSERVED_LOG2_PATH,
    index=True,
)

observed_mg_l_matrix.to_csv(
    OBSERVED_MG_L_PATH,
    index=True,
)

structural_log2_matrix.to_csv(
    STRUCTURAL_LOG2_PATH,
    index=True,
)

chromosome_table.to_csv(
    CHROMOSOME_PATH,
    index=False,
)

plasmid_table.to_csv(
    PLASMID_PATH,
    index=False,
)

print("=" * 90)
print("CELL 5 — COMPLETE 200 x 21 DATASET QC")
print("=" * 90)
print(f"Observation rows:             {X.shape[0]:,}")
print(f"Fixed-effect columns:         {X.shape[1]:,}")
print(f"Design-matrix rank:           {np.linalg.matrix_rank(X):,}")
print(f"Targeted chromosome matrix:   {C.shape}")
print(f"Plasmid feature matrix:       {P.shape}")
print(f"K dimensions:                 {K.shape}")
print(f"K mean diagonal:              {np.mean(np.diag(K)):.6f}")
print(f"K minimum eigenvalue:         {K_eigenvalues.min():.6e}")
print(f"Observed log2 MIC mean:        {np.mean(y):.6f}")
print(f"Observed log2 MIC SD:          {np.std(y, ddof=1):.6f}")
print(f"Reference CN_TEM1:             {CN_REFERENCE:.6f}")

print("\nSampled plasmid promoter/CN profiles:")
display(
    plasmid_table[
        [
            "plasmid_id",
            "sutcliffe_32_nt",
            "sutcliffe_162_nt",
            "sutcliffe_175_nt",
            "CN_TEM1",
            "q_CN",
        ]
    ]
)

print("\nFirst five rows of the complete log2 MIC matrix:")
display(
    observed_log2_matrix.head()
)

print("\nSaved:")
for path in [
    OBSERVED_LOG2_PATH,
    OBSERVED_MG_L_PATH,
    STRUCTURAL_LOG2_PATH,
    CHROMOSOME_PATH,
    PLASMID_PATH,
]:
    print(path)

print("\nCell 5: PASS")


In [ ]:
#@title Cell 6 - Fit the theoretical model by REML and GLS

start_time = time.time()

example_static = prepare_reml_static(
    example["X"],
    example["pathogen_index"],
    example["K"],
)

example_fit = fit_section2_reml_gls(
    example["X"],
    example["y"],
    example["pathogen_index"],
    example["K"],
    static=example_static,
)

elapsed = time.time() - start_time

(
    alpha_hat,
    beta_C_hat,
    beta_P_hat,
    B_hat,
) = unpack_beta(
    example_fit["beta_hat"]
)

print("=" * 90)
print("CELL 6 — REML/GLS FIT")
print("=" * 90)
print(f"True alpha:                  {ALPHA_TRUE:.6f}")
print(f"Estimated alpha:             {alpha_hat:.6f}")
print(f"True sigma_g^2:              {SIGMA_G2_TRUE:.6f}")
print(f"Estimated sigma_g^2:         {example_fit['sigma_g2_hat']:.6f}")
print(f"True sigma_e^2:              {SIGMA_E2_TRUE:.6f}")
print(f"Estimated sigma_e^2:         {example_fit['sigma_e2_hat']:.6f}")
print(f"Estimated variance ratio:    {example_fit['delta_hat']:.6f}")
print(f"REML optimizer evaluations:  {example_fit['optimization_nfev']}")
print(f"Fit time:                    {elapsed:.2f} seconds")

plasmid_coefficient_comparison = pd.DataFrame({
    "feature": PLASMID_FEATURE_LABELS,
    "true_beta_P": BETA_P_TRUE,
    "estimated_beta_P": beta_P_hat,
})

print("\nPlasmid main-effect recovery:")
display(plasmid_coefficient_comparison)

print("\nCell 6: PASS")


In [ ]:
#@title Cell 7 - Evaluate recovery of Delta and DeltaDelta

def true_and_estimated_effects(dataset, fit):
    C = dataset["C"]
    P = dataset["P"]

    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        fit["beta_hat"]
    )

    # Structural truth: u_i cancels within pathogen and E[epsilon]=0.
    y0_true = (
        ALPHA_TRUE
        + C @ dataset["beta_C_true"]
    )

    delta_true = (
        P @ dataset["beta_P_true"]
    )[None, :] + (
        C
        @ dataset["B_true"]
        @ P.T
    )

    yij_true = (
        y0_true[:, None]
        + delta_true
    )

    y0_hat = (
        alpha_hat
        + C @ beta_C_hat
    )

    delta_hat = (
        P @ beta_P_hat
    )[None, :] + (
        C
        @ B_hat
        @ P.T
    )

    yij_hat = (
        y0_hat[:, None]
        + delta_hat
    )

    return {
        "y0_true": y0_true,
        "y0_hat": y0_hat,
        "yij_true": yij_true,
        "yij_hat": yij_hat,
        "delta_true": delta_true,
        "delta_hat": delta_hat,
    }


def basic_metrics(
    true_values,
    estimated_values,
):
    true_values = np.asarray(
        true_values,
        dtype=float,
    ).ravel()

    estimated_values = np.asarray(
        estimated_values,
        dtype=float,
    ).ravel()

    error = (
        estimated_values
        - true_values
    )

    nonzero = (
        np.abs(true_values)
        > 1e-12
    )

    if nonzero.any():
        sign_accuracy = np.mean(
            np.sign(
                estimated_values[nonzero]
            )
            == np.sign(
                true_values[nonzero]
            )
        )
    else:
        sign_accuracy = np.nan

    return {
        "bias": float(
            np.mean(error)
        ),
        "rmse": float(
            np.sqrt(
                np.mean(error ** 2)
            )
        ),
        "sign_accuracy": float(
            sign_accuracy
        ),
    }


def pairwise_delta_delta(
    delta_matrix,
):
    """
    DeltaDelta for all unordered pathogen pairs and all plasmids.
    Pair orientation is i < k.
    """
    upper_i, upper_k = np.triu_indices(
        delta_matrix.shape[0],
        k=1,
    )

    dd = (
        delta_matrix[upper_i, :]
        - delta_matrix[upper_k, :]
    )

    return (
        dd,
        upper_i,
        upper_k,
    )


def evaluate_dataset_fit(
    dataset,
    fit,
):
    effects = true_and_estimated_effects(
        dataset,
        fit,
    )

    (
        dd_true,
        upper_i,
        upper_k,
    ) = pairwise_delta_delta(
        effects["delta_true"]
    )

    dd_hat, _, _ = (
        pairwise_delta_delta(
            effects["delta_hat"]
        )
    )

    metrics_y0 = basic_metrics(
        effects["y0_true"],
        effects["y0_hat"],
    )

    metrics_yij = basic_metrics(
        effects["yij_true"],
        effects["yij_hat"],
    )

    metrics_delta = basic_metrics(
        effects["delta_true"],
        effects["delta_hat"],
    )

    metrics_dd = basic_metrics(
        dd_true,
        dd_hat,
    )

    row = {
        "y0_bias": metrics_y0["bias"],
        "y0_rmse": metrics_y0["rmse"],
        "yij_bias": metrics_yij["bias"],
        "yij_rmse": metrics_yij["rmse"],
        "delta_bias": metrics_delta["bias"],
        "delta_rmse": metrics_delta["rmse"],
        "delta_sign_accuracy": metrics_delta["sign_accuracy"],
        "delta_delta_bias": metrics_dd["bias"],
        "delta_delta_rmse": metrics_dd["rmse"],
        "delta_delta_sign_accuracy": metrics_dd["sign_accuracy"],
        "sigma_g2_hat": fit["sigma_g2_hat"],
        "sigma_e2_hat": fit["sigma_e2_hat"],
        "delta_variance_ratio_hat": fit["delta_hat"],
    }

    return (
        row,
        effects,
        dd_true,
        dd_hat,
        upper_i,
        upper_k,
    )


(
    example_metrics,
    example_effects,
    example_dd_true,
    example_dd_hat,
    example_pair_i,
    example_pair_k,
) = evaluate_dataset_fit(
    example,
    example_fit,
)

display(
    pd.DataFrame(
        [example_metrics]
    ).T.rename(
        columns={0: "value"}
    )
)

delta_true_df = pd.DataFrame(
    example_effects["delta_true"],
    index=pathogen_labels,
    columns=[
        f"P{j}"
        for j in range(1, N_PLASMIDS + 1)
    ],
)

delta_hat_df = pd.DataFrame(
    example_effects["delta_hat"],
    index=pathogen_labels,
    columns=[
        f"P{j}"
        for j in range(1, N_PLASMIDS + 1)
    ],
)

DELTA_TRUE_PATH = (
    OUTPUT_DIR
    / "01_example_true_Delta.csv"
)

DELTA_HAT_PATH = (
    OUTPUT_DIR
    / "01_example_estimated_Delta.csv"
)

delta_true_df.to_csv(
    DELTA_TRUE_PATH,
    index=True,
)

delta_hat_df.to_csv(
    DELTA_HAT_PATH,
    index=True,
)

print("\nSaved:")
print(DELTA_TRUE_PATH)
print(DELTA_HAT_PATH)
print("\nCell 7: PASS")


In [ ]:
#@title Cell 8 - Run 100 independent complete-design simulation replicates

def run_one_complete_replicate(
    replicate_index,
):
    seed = (
        MASTER_SEED
        + 1000
        + int(replicate_index)
    )

    dataset = simulate_complete_dataset(
        seed
    )

    static = prepare_reml_static(
        dataset["X"],
        dataset["pathogen_index"],
        dataset["K"],
    )

    fit = fit_section2_reml_gls(
        dataset["X"],
        dataset["y"],
        dataset["pathogen_index"],
        dataset["K"],
        static=static,
    )

    (
        metrics,
        _,
        _,
        _,
        _,
        _,
    ) = evaluate_dataset_fit(
        dataset,
        fit,
    )

    metrics["replicate"] = int(
        replicate_index + 1
    )
    metrics["seed"] = int(seed)

    return metrics


replicate_rows = []

start_time = time.time()

for r in range(
    N_SIM_REPLICATES
):
    replicate_rows.append(
        run_one_complete_replicate(r)
    )

    if (
        (r + 1) % 10 == 0
        or r == 0
        or r + 1 == N_SIM_REPLICATES
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {r + 1:3d}/{N_SIM_REPLICATES} replicates "
            f"({elapsed:.1f} seconds elapsed)"
        )

replicate_results = pd.DataFrame(
    replicate_rows
)

REPLICATE_RESULTS_PATH = (
    OUTPUT_DIR
    / "01_scenario1_updated_complete_design_replicate_metrics.csv"
)

replicate_results.to_csv(
    REPLICATE_RESULTS_PATH,
    index=False,
)

print("\nSaved:")
print(REPLICATE_RESULTS_PATH)
print("\nCell 8: PASS")


In [ ]:
#@title Cell 9 - Summarize the 100-replicate benchmark

PRIMARY_METRICS = [
    "delta_bias",
    "delta_rmse",
    "delta_sign_accuracy",
    "delta_delta_bias",
    "delta_delta_rmse",
    "delta_delta_sign_accuracy",
]

SUPPORTING_METRICS = [
    "y0_bias",
    "y0_rmse",
    "yij_bias",
    "yij_rmse",
    "sigma_g2_hat",
    "sigma_e2_hat",
]

summary_rows = []

for metric in (
    PRIMARY_METRICS
    + SUPPORTING_METRICS
):
    values = replicate_results[
        metric
    ].to_numpy(
        dtype=float
    )

    summary_rows.append({
        "metric": metric,
        "mean": float(
            np.nanmean(values)
        ),
        "sd": float(
            np.nanstd(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.nanmedian(values)
        ),
        "q025": float(
            np.nanquantile(
                values,
                0.025,
            )
        ),
        "q975": float(
            np.nanquantile(
                values,
                0.975,
            )
        ),
    })

scenario1_summary = pd.DataFrame(
    summary_rows
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "01_scenario1_updated_complete_design_summary.csv"
)

scenario1_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print("=" * 90)
print("SCENARIO 1 — UPDATED 100-REPLICATE SUMMARY")
print("=" * 90)

display(
    scenario1_summary[
        scenario1_summary[
            "metric"
        ].isin(
            PRIMARY_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nSupporting diagnostics:")

display(
    scenario1_summary[
        scenario1_summary[
            "metric"
        ].isin(
            SUPPORTING_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nSaved:")
print(SUMMARY_PATH)
print("\nCell 9: PASS")


In [ ]:
#@title Cell 10 - Parametric bootstrap for the representative dataset

def simulate_parametric_bootstrap_y(
    rng,
    X,
    pathogen_index,
    K,
    beta_hat,
    sigma_g2_hat,
    sigma_e2_hat,
):
    u_star = draw_correlated_host_effect(
        rng,
        K,
        sigma_g2_hat,
    )

    epsilon_star = rng.normal(
        0.0,
        np.sqrt(
            sigma_e2_hat
        ),
        size=X.shape[0],
    )

    y_star = (
        X @ beta_hat
        + u_star[
            pathogen_index
        ]
        + epsilon_star
    )

    return y_star


def effects_from_beta(
    C,
    P,
    beta_hat,
):
    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        beta_hat
    )

    y0_hat = (
        alpha_hat
        + C @ beta_C_hat
    )

    delta_hat = (
        P @ beta_P_hat
    )[None, :] + (
        C
        @ B_hat
        @ P.T
    )

    yij_hat = (
        y0_hat[:, None]
        + delta_hat
    )

    return (
        y0_hat,
        yij_hat,
        delta_hat,
    )


n_pairs = len(
    example_pair_i
)

n_delta = (
    N_PATHOGENS
    * N_PLASMIDS
)

n_delta_delta = (
    n_pairs
    * N_PLASMIDS
)

bootstrap_delta = np.empty(
    (
        N_BOOTSTRAP,
        n_delta,
    ),
    dtype=np.float32,
)

bootstrap_delta_delta = np.empty(
    (
        N_BOOTSTRAP,
        n_delta_delta,
    ),
    dtype=np.float32,
)

bootstrap_rng = np.random.default_rng(
    MASTER_SEED + 900000
)

start_time = time.time()

for b in range(
    N_BOOTSTRAP
):
    y_star = simulate_parametric_bootstrap_y(
        bootstrap_rng,
        example["X"],
        example["pathogen_index"],
        example["K"],
        example_fit["beta_hat"],
        example_fit["sigma_g2_hat"],
        example_fit["sigma_e2_hat"],
    )

    fit_star = fit_section2_reml_gls(
        example["X"],
        y_star,
        example["pathogen_index"],
        example["K"],
        static=example_static,
    )

    _, _, delta_star = effects_from_beta(
        example["C"],
        example["P"],
        fit_star["beta_hat"],
    )

    dd_star, _, _ = (
        pairwise_delta_delta(
            delta_star
        )
    )

    bootstrap_delta[
        b,
        :,
    ] = delta_star.ravel().astype(
        np.float32
    )

    bootstrap_delta_delta[
        b,
        :,
    ] = dd_star.ravel().astype(
        np.float32
    )

    if (
        (b + 1) % 20 == 0
        or b == 0
        or b + 1 == N_BOOTSTRAP
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {b + 1:3d}/{N_BOOTSTRAP} bootstrap refits "
            f"({elapsed:.1f} seconds elapsed)"
        )

delta_ci_low = np.quantile(
    bootstrap_delta,
    0.025,
    axis=0,
)

delta_ci_high = np.quantile(
    bootstrap_delta,
    0.975,
    axis=0,
)

dd_ci_low = np.quantile(
    bootstrap_delta_delta,
    0.025,
    axis=0,
)

dd_ci_high = np.quantile(
    bootstrap_delta_delta,
    0.975,
    axis=0,
)

delta_true_flat = (
    example_effects[
        "delta_true"
    ].ravel()
)

dd_true_flat = (
    example_dd_true.ravel()
)

delta_excludes_zero = (
    (delta_ci_low > 0)
    | (delta_ci_high < 0)
)

dd_excludes_zero = (
    (dd_ci_low > 0)
    | (dd_ci_high < 0)
)

delta_contains_truth = (
    (delta_ci_low <= delta_true_flat)
    & (
        delta_true_flat
        <= delta_ci_high
    )
)

dd_contains_truth = (
    (dd_ci_low <= dd_true_flat)
    & (
        dd_true_flat
        <= dd_ci_high
    )
)

bootstrap_summary = pd.DataFrame([
    {
        "effect": "Delta_ij",
        "number_of_effects": int(
            len(delta_true_flat)
        ),
        "fraction_CI_excludes_zero": float(
            np.mean(
                delta_excludes_zero
            )
        ),
        "fraction_CI_contains_known_truth": float(
            np.mean(
                delta_contains_truth
            )
        ),
    },
    {
        "effect": "DeltaDelta_ikj",
        "number_of_effects": int(
            len(dd_true_flat)
        ),
        "fraction_CI_excludes_zero": float(
            np.mean(
                dd_excludes_zero
            )
        ),
        "fraction_CI_contains_known_truth": float(
            np.mean(
                dd_contains_truth
            )
        ),
    },
])

BOOTSTRAP_SUMMARY_PATH = (
    OUTPUT_DIR
    / "01_scenario1_updated_representative_bootstrap_summary.csv"
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

display(
    bootstrap_summary
)

print("\nImportant:")
print(
    "The interval-containment values summarize effects in this representative "
    "simulated dataset. Formal repeated-dataset coverage requires repeating "
    "the bootstrap across independent simulation datasets."
)

print("\nSaved:")
print(BOOTSTRAP_SUMMARY_PATH)
print("\nCell 10: PASS")


In [ ]:
#@title Cell 11 - Optional repeated-dataset bootstrap coverage

def bootstrap_selected_effects_for_dataset(
    dataset,
    fit,
    n_bootstrap,
    rng_seed,
):
    """
    Optional formal coverage for two pre-specified effects:
      1) Delta for pathogen 0 with plasmid 0
      2) DeltaDelta for pathogen 0 versus pathogen 1 with plasmid 0
    """
    rng = np.random.default_rng(
        rng_seed
    )

    static = prepare_reml_static(
        dataset["X"],
        dataset["pathogen_index"],
        dataset["K"],
    )

    effects = true_and_estimated_effects(
        dataset,
        fit,
    )

    true_delta = float(
        effects["delta_true"][
            0,
            0,
        ]
    )

    true_dd = float(
        effects["delta_true"][
            0,
            0,
        ]
        - effects["delta_true"][
            1,
            0,
        ]
    )

    boot_delta = np.empty(
        n_bootstrap,
        dtype=float,
    )

    boot_dd = np.empty(
        n_bootstrap,
        dtype=float,
    )

    for b in range(
        n_bootstrap
    ):
        y_star = simulate_parametric_bootstrap_y(
            rng,
            dataset["X"],
            dataset["pathogen_index"],
            dataset["K"],
            fit["beta_hat"],
            fit["sigma_g2_hat"],
            fit["sigma_e2_hat"],
        )

        fit_star = fit_section2_reml_gls(
            dataset["X"],
            y_star,
            dataset["pathogen_index"],
            dataset["K"],
            static=static,
        )

        _, _, delta_star = effects_from_beta(
            dataset["C"],
            dataset["P"],
            fit_star["beta_hat"],
        )

        boot_delta[b] = (
            delta_star[
                0,
                0,
            ]
        )

        boot_dd[b] = (
            delta_star[
                0,
                0,
            ]
            - delta_star[
                1,
                0,
            ]
        )

    delta_low, delta_high = np.quantile(
        boot_delta,
        [0.025, 0.975],
    )

    dd_low, dd_high = np.quantile(
        boot_dd,
        [0.025, 0.975],
    )

    return {
        "true_delta": true_delta,
        "delta_low": float(delta_low),
        "delta_high": float(delta_high),
        "delta_contains_truth": bool(
            delta_low
            <= true_delta
            <= delta_high
        ),
        "true_delta_delta": true_dd,
        "delta_delta_low": float(dd_low),
        "delta_delta_high": float(dd_high),
        "delta_delta_contains_truth": bool(
            dd_low
            <= true_dd
            <= dd_high
        ),
    }


if RUN_FULL_BOOTSTRAP_COVERAGE:
    coverage_rows = []
    coverage_start = time.time()

    for r in range(
        N_SIM_REPLICATES
    ):
        seed = (
            MASTER_SEED
            + 1000
            + r
        )

        dataset = simulate_complete_dataset(
            seed
        )

        static = prepare_reml_static(
            dataset["X"],
            dataset["pathogen_index"],
            dataset["K"],
        )

        fit = fit_section2_reml_gls(
            dataset["X"],
            dataset["y"],
            dataset["pathogen_index"],
            dataset["K"],
            static=static,
        )

        row = bootstrap_selected_effects_for_dataset(
            dataset,
            fit,
            N_BOOTSTRAP,
            MASTER_SEED
            + 2_000_000
            + r,
        )

        row["replicate"] = r + 1
        row["seed"] = seed
        coverage_rows.append(row)

        if (
            (r + 1) % 10 == 0
            or r == 0
            or r + 1 == N_SIM_REPLICATES
        ):
            print(
                f"Coverage replicate {r + 1}/{N_SIM_REPLICATES}"
            )

    coverage_results = pd.DataFrame(
        coverage_rows
    )

    COVERAGE_PATH = (
        OUTPUT_DIR
        / "01_scenario1_updated_formal_bootstrap_coverage.csv"
    )

    coverage_results.to_csv(
        COVERAGE_PATH,
        index=False,
    )

    display(
        coverage_results[
            [
                "delta_contains_truth",
                "delta_delta_contains_truth",
            ]
        ].mean().to_frame(
            "coverage"
        )
    )

    print("\nSaved:")
    print(COVERAGE_PATH)

    print(
        f"Elapsed formal coverage time: "
        f"{(time.time() - coverage_start) / 60:.1f} minutes"
    )

else:
    print(
        "Formal repeated-dataset bootstrap coverage is disabled because "
        "RUN_FULL_BOOTSTRAP_COVERAGE=False."
    )
    print(
        "This remains optional until the main updated Scenario 1 benchmark "
        "and representative bootstrap have been checked."
    )

print("\nCell 11: PASS")


In [ ]:
#@title Cell 12 - Final QC and output manifest

required_output_files = [
    OBSERVED_LOG2_PATH,
    OBSERVED_MG_L_PATH,
    STRUCTURAL_LOG2_PATH,
    CHROMOSOME_PATH,
    PLASMID_PATH,
    DELTA_TRUE_PATH,
    DELTA_HAT_PATH,
    REPLICATE_RESULTS_PATH,
    SUMMARY_PATH,
    BOOTSTRAP_SUMMARY_PATH,
]

missing_outputs = [
    str(path)
    for path in required_output_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Required updated Scenario 1 output(s) are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )

manifest = {
    "notebook":
        "07_Simulation_01_Complete_Design_UPDATED.ipynb",

    "scenario":
        "Simulation 01 - complete design - biologically constrained update",

    "central_question":
        "Does the effect of plasmid j depend on the pathogen chromosome i?",

    "model": (
        "y_ij = alpha + beta_C^T c_i + beta_P^T p_j "
        "+ c_i^T B p_j + u_i + epsilon_ij"
    ),

    "simulation_settings": {
        "pathogens":
            N_PATHOGENS,

        "plasmids":
            N_PLASMIDS,

        "plasmid_states_including_P0":
            N_PLASMIDS + 1,

        "complete_MIC_observations":
            N_PATHOGENS
            * (N_PLASMIDS + 1),

        "predefined_genes":
            PREDEFINED_GENES,

        "gene_centered_units":
            len(PREDEFINED_GENES),

        "chromosomal_features":
            D_C,

        "chromosomal_state_values":
            CHROMOSOMAL_STATES.tolist(),

        "chromosomal_state_probabilities":
            CHROMOSOMAL_STATE_PROBS.tolist(),

        "background_snps":
            N_BACKGROUND_SNPS,

        "plasmid_features":
            PLASMID_FEATURE_LABELS,

        "empirical_plasmid_pair_source_promoter":
            str(PROMOTER_FILE),

        "empirical_plasmid_pair_source_copy_number":
            str(CN_FILE),

        "empirical_CN_reference_median":
            CN_REFERENCE,

        "beta_P":
            {
                label: float(value)
                for label, value in zip(
                    PLASMID_FEATURE_LABELS,
                    BETA_P_TRUE,
                )
            },

        "interaction_magnitude":
            INTERACTION_MAGNITUDE,

        "interaction_plasmid_component":
            "TEM1_plasmid_presence_only",

        "alpha":
            ALPHA_TRUE,

        "sigma_g":
            SIGMA_G_TRUE,

        "sigma_e":
            SIGMA_E_TRUE,

        "sigma_g2":
            SIGMA_G2_TRUE,

        "sigma_e2":
            SIGMA_E2_TRUE,

        "simulation_replicates":
            N_SIM_REPLICATES,

        "bootstrap_replicates":
            N_BOOTSTRAP,
    },

    "primary_targets": [
        "Delta_ij = y_ij - y_i0",
        "DeltaDelta_ikj = Delta_ij - Delta_kj",
    ],

    "primary_metrics": [
        "delta_bias",
        "delta_rmse",
        "delta_delta_bias",
        "delta_delta_rmse",
    ],

    "secondary_metrics": [
        "delta_sign_accuracy",
        "delta_delta_sign_accuracy",
    ],

    "outputs": [
        str(path)
        for path in required_output_files
    ],
}

MANIFEST_PATH = (
    OUTPUT_DIR
    / "01_scenario1_updated_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

print("=" * 90)
print("UPDATED SCENARIO 1 COMPLETE")
print("=" * 90)
print(f"Manifest: {MANIFEST_PATH}")
print(f"Required output files checked: {len(required_output_files)}")
print("\nThe old generic random-effect biological specification is not used in this notebook.")
print("Cell 12: PASS")
